<center><h1>Satish_Manvika_HW3</h1></center>
<br>
<br>

Name: Manvika Satish
<br>
Github Username: manvsatish
<br>
USC ID: 5582061880

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [2]:
import io
import re
import zipfile
import numpy as np
import pandas as pd

from pathlib import Path

Get the AReM Data Set

In [3]:
zip_path = Path("../data/AReM.zip")
print("zip file exists: ", zip_path.exists())
print("relative path: ", zip_path)
assert zip_path.exists(), f"Data file not found at {zip_path}"

zip file exists:  True
relative path:  ../data/AReM.zip


In [4]:
with zipfile.ZipFile(zip_path, "r") as archive:
    csv_files = sorted(name for name in archive.namelist() if name.endswith(".csv") and "__MACOSX" not in name)

print("Number of CSV files: ", len(csv_files))
print("first five files:")
for name in csv_files[:5]:
    print(name)

assert len(csv_files) == 88, f"Expected 88 csv files, found {len(csv_files)}"

Number of CSV files:  88
first five files:
AReM/bending1/dataset1.csv
AReM/bending1/dataset2.csv
AReM/bending1/dataset3.csv
AReM/bending1/dataset4.csv
AReM/bending1/dataset5.csv


In [6]:
column_names = ["time", "avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23",]
expected_times = np.arange(0, 120000, 250)


def read_activity_csv(archive, member_name):
    text = archive.read(member_name).decode("utf-8", errors="replace") #read and clean one arem csv file from zip archive
    rows = []

    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        parts = line.split(",") if "," in line else line.split() # most files use commas but bending2/dataset4 uses whitespace
        parts = [part.strip() for part in parts if part.strip()] # remove empty fields caused by trailing commas

        if len(parts) < 7:
            continue

        try:
            rows.append([float(value) for value in parts[:7]])
        except ValueError:
            continue #ignore any non-num header row

    data = pd.DataFrame(rows, columns=column_names)
    original_rows = len(data)
    data["time"] = data["time"].astype(int)
    data = data.groupby("time", as_index=False).mean()
    missing_times = sorted(set(expected_times) - set(data["time"]))

    needs_repair = (len(data) != len(expected_times) or not np.array_equal(data["time"].to_numpy(), expected_times))
    if needs_repair: # sitting/dataset8 is missing timestamp 13500
        data = (data.set_index("time").reindex(expected_times).interpolate(method="linear", limit_direction="both").rename_axis("time").reset_index())

    assert data.shape == (480, 7), (f"{member_name} has unexpected shape {data.shape}")
    assert not data.isna().any().any(), (f"{member_name} still contains missing values")
    cleaning_info = {"file": member_name, "original_rows": original_rows,"final_rows": len(data), "missing_times": missing_times, "repaired": needs_repair,}

    return data, cleaning_info

In [7]:
instances = []
cleaning_records = []

with zipfile.ZipFile(zip_path, "r") as archive:
    for member_name in csv_files:
        activity = Path(member_name).parent.name
        dataset_number = int(re.search(r"dataset(\d+)\.csv$", member_name).group(1))
        data, cleaning_info = read_activity_csv(archive, member_name)

        #for bending activities, datasets 1 and 2 are test data
        # for all other activities, datasets 1, 2, and 3 are test data.
        if activity in {"bending1", "bending2"}:
            split = "test" if dataset_number in {1, 2} else "train"
        else:
            split = "test" if dataset_number in {1, 2, 3} else "train"

        instances.append({"activity": activity, "dataset": dataset_number, "split": split, "data": data,})
        cleaning_info.update({"activity": activity, "dataset": dataset_number,})
        cleaning_records.append(cleaning_info)

print("total instances loaded:", len(instances))
assert len(instances) == 88

total instances loaded: 88


In [8]:
split_summary = (pd.DataFrame(instances).groupby(["activity", "split"]).size().unstack(fill_value=0).sort_index())
split_summary.loc["Total"] = split_summary.sum()
split_summary

split,test,train
activity,,
bending1,2,5
bending2,2,4
cycling,3,12
lying,3,12
sitting,3,12
standing,3,12
walking,3,12
Total,19,69


In [9]:
cleaning_summary = pd.DataFrame(cleaning_records)
cleaning_summary.loc[cleaning_summary["repaired"], ["activity","dataset","original_rows","final_rows","missing_times",],]

,activity,dataset,original_rows,final_rows,missing_times
56,sitting,8,479,480,[13500]


### Data loading and cleaning notes

The ZIP archive has 88 CSV files. I was able to spot some formatting inconsistencies (through Piazza) and handled the issues during loading:
- `bending2/dataset4.csv` is whitespace-delimited, not comma-delimited.
- `cycling/dataset9.csv` has an extra trailing comma on the final data row.
- `cycling/dataset14.csv` has an extra trailing comma on the final data row.
- `sitting/dataset8.csv` has 479 observations and is missing timestamp 13500. The missing observation was restored by linear interpolation between its adjacent time points.

After cleaning, every instance has 480 observations and seven columns. The resulting split contains 69 training instances and 19 test instances.

### (b) Test and Train Data

### (c) Feature Extraction

#### i. Research

#### ii. Extraction

#### iii. Standard Deviation

#### iv. Select Features

## 2. ISLR 3.7.4

### (a) Linear Train

### (b) Linear Test

### (c) Not Linear Train

### (d) Not Linear Testing

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 